# Import

In [ ]:
from scipy.io import loadmat
import pandas as pd
import os
import matplotlib.pyplot as plt
import numpy as np
from mapd import Trial, Table, Sinq
import h5py
%load_ext autoreload
%autoreload 2
%matplotlib widget

def refresh_table_addons():
    import importlib
    import mapd.table_plotters as tps
    import mapd.table_scalars as tbs
    importlib.reload(tps)
    importlib.reload(tbs)

    for name in dir(tps):
        if name.startswith('_'):
            continue
        attr = getattr(tps, name)
        setattr(Table, name[len("plot_"):], attr)
    
    for name in dir(tbs):
        if name.startswith('_'):
            continue
        if name.startswith('compute_'):
            attr = getattr(tbs, name)
            setattr(Table, name[len("compute_"):], attr)

# Figure 1

# Create Figure 1 sinq

In [ ]:
sinq = Sinq(sinqname='Figure1')
print(sinq.__repr__())
sinq.df

## Figure 1D example traces

In [ ]:
from mapd.table_export_methods import plot_probe_trace_from_pickle

# 250304_F3_D1
T = sinq.df.loc['250304_F3_C1','Table']
idx = T.df.loc[23:23].index
%matplotlib inline
# fig,ax = T.plot_some_trials(index=idx, from_zero=True)
# fig.canvas.draw()
# fig
fname = T.export_some_probe_groups(index=idx, from_zero=True)
fig = plot_probe_trace_from_pickle(fname)
fig.canvas.draw()
figname =  f"./Figure1/example_trace_{T.day}_F{T.fly}_C{T.cell}_{T.genotype}_T{idx[0]}_{idx[-1]}.svg"
fig.savefig(fname=figname, format='svg')
fig



In [ ]:
from mapd.table_export_methods import plot_probe_trace_from_pickle

# 250304_F3_D1
idx = T.df.loc[20:38].index
%matplotlib inline
# fig,ax = T.plot_some_trials(index=idx, from_zero=True)
# fig.canvas.draw()
# fig
fname = T.export_some_probe_groups(index=idx, from_zero=True)
fig = plot_probe_trace_from_pickle(fname)
fig.canvas.draw()
figname =  f"./Figure1/example_trace_{T.day}_F{T.fly}_C{T.cell}_{T.genotype}_T{idx[0]}_{idx[-1]}.svg"
fig.savefig(fname=figname, format='svg')
fig




# Figure 1E - heatmaps

### Red

In [ ]:
T = sinq.df.loc['210405_F1_C1','Table'] # first one
T.fig_folder = './Figure1'
fig,ax,df = T.plot_probe_position_heatmap(index=T.df.loc[51:450].index,cmin=-500,cmax=10,format='svg')
fig

### Blue

In [ ]:
T = sinq.df.loc['250304_F3_C1','Table'] # long one
T.fig_folder = './Figure1'
fig,ax,df = T.plot_probe_position_heatmap(index=T.df.loc[1:400].index, cmin=-500,cmax=10,format='svg')
fig

### Green

In [ ]:
T = sinq.df.loc['250228_F2_C1','Table'] # first one
T.fig_folder = './Figure1'
fig,ax,df = T.plot_probe_position_heatmap(index=T.df.loc[101:500].index,cmin=-500,cmax=10,format='svg')
fig

# Figure 1F - probe distributions

### Run export functions for all tables

In [ ]:
refresh_table_addons()

# export the entire distribution
def call_and_export_probe_distributions(row):
    T = sinq.restore_table(row.name)
    T.fig_folder = './Figure1'
    print('Table: {}'.format(T))
    T.plot_probe_distribution(df_filter=None, index=None, bin_min=-500, bin_max=10,export=True)
    T.plot_probe_distribution(df_filter = {'pyasState':'hi'}, index=None, bin_min=-500, bin_max=10,export=True)
    T.plot_probe_distribution(df_filter = {'pyasState':'lo'}, index=None, bin_min=-500, bin_max=10,export=True)

remaining = sinq.df.loc['250506_F2_C1':'250923_F2_C1'] 
for dfc in remaining.index:
    row = sinq.df.loc[dfc].copy()
    call_and_export_probe_distributions(row)
    sinq.drop_tables()


## Import and gather histograms

In [ ]:
import glob
import numpy as np
import pickle
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.figure import Figure
from matplotlib.backends.backend_agg import FigureCanvasAgg as FigureCanvas
import matplotlib as mpl
mpl.rcParams.update(mpl.rcParamsDefault)  # reset to defaults
mpl.rcParams['pdf.fonttype'] = 42         # embed fonts as text, not paths
mpl.rcParams['svg.fonttype'] = 'none'     # keep text editable in SVG
mpl.rcParams['font.family'] = 'Arial'
mpl.rcParams['font.size'] = 11

import random

folder_path = "./Figure1/exports"
fig_folder = './Figure1'

    # Color scheme for targets, adjust as needed
_force_clrs = [
    (np.float64(0.95447591), np.float64(0.47082238), np.float64(0.32310953)),
    (np.float64(0.7965014), np.float64(0.10506637), np.float64(0.31063031)),
    ]

def ave_and_plot_histograms(dfcs, pat,targets=None,title='None'):
    fig = Figure(figsize=(6,6), dpi=300)
    canvas = FigureCanvas(fig)
    ax = fig.add_subplot(111)

    if targets:
        for key in targets.keys():
            targ_dict = targets[key]
            tgt_clr = _force_clrs[0 if targ_dict['pyasState']=='lo' else 1]
            rect = patches.Rectangle(
                (0, targ_dict['pyasXPosition']),  # (x,y) lower-left corner
                .1,                # width across counts max
                targ_dict['pyasWidth'],            # height
                edgecolor=tgt_clr,
                facecolor=tgt_clr,
                alpha=1
            )
            ax.add_patch(rect)

    normalized_counts_list = []
    probe_bins = None

    for dfc in dfcs:
        pattern=f'{dfc}'+pat
        print(pattern)
        file_path = glob.glob(os.path.join(folder_path, pattern))

        with open(file_path[0], 'rb') as f:
            data = pickle.load(f)
            bins = data['probe_bins']
            
            if probe_bins is None:
                probe_bins = bins
            else:
                assert np.allclose(probe_bins, bins), "Probe bins differ between files!"

            counts = data['total_counts']
            if '210302_F1_C2' in file_path[0]:
                print('shifting the bins for this one fly')
                print(type(counts))
                print(counts.shape)
                counts = np.roll(counts,-10,axis=0)

            norm_counts = counts / counts.sum()  # Normalize to sum to 1
            normalized_counts_list.append(norm_counts)
            ax.step(norm_counts, probe_bins[:-1], where='post', label=dfc)

    avg_norm_counts = np.mean(normalized_counts_list, axis=0)
    ax.step(avg_norm_counts, probe_bins[:-1], where='post', color='black', linewidth=2, label='Avg Normalized Histogram')

    ax.legend()
    ax.set_xlabel('Probe Position')
    ax.set_ylabel('Normalized Frequency')
    ax.grid(True, alpha=0.5)

    save_path = os.path.join(fig_folder, title + '.svg')
    fig.savefig(save_path,format = 'svg')
    print(f"Saved plot to {save_path}")
    return fig


### Red LED Hot-cell flies

In [ ]:
T = sinq.restore_table(dayflycell='210917_F2_C1') # last one
targets = T.targets
T.fig_folder = 'Figure1'
red_mask = (sinq.df['most_common_fiberLED']=='625_red')
dfcs = sinq.df.loc[red_mask].index
figs = []

for pat,fn in zip(['*_probe_dist_lo_*.pkl','*_probe_dist_hi_*.pkl','*_probe_dist_nofilt_*.pkl'],
                  ['probe_dist_lo','probe_dist_hi','probe_dist_nofilt']):
    figs.append(ave_and_plot_histograms(dfcs=dfcs,pat=pat,targets=targets,title= fn+ '_red_hc'))

### Blue flies

In [ ]:
T = sinq.restore_table(dayflycell='250925_F2_C1')
targets = T.targets
T.fig_folder = 'Figure1'
blue_mask = (sinq.df['blue_toggle_fraction']==1) & ~(sinq.df.index.values=='250304_F2_C1') # Right now, this excludes the frustrated green fly (sinq.df['blue_fraction']>.5)
dfcs = sinq.df.loc[blue_mask].index
figs = []
for pat,fn in zip(['*_probe_dist_lo_*.pkl','*_probe_dist_hi_*.pkl','*_probe_dist_nofilt_*.pkl'],
                  ['probe_dist_lo','probe_dist_hi','probe_dist_nofilt']):
    figs.append(ave_and_plot_histograms(dfcs=dfcs,pat=pat,targets=targets,title=fn+'_blue_green_wt'))



### Green flies

In [ ]:
targets = T.targets
green_mask = (sinq.df['blue_toggle_fraction']==1) & (sinq.df['blue_fraction']<=.5)
dfcs = sinq.df.loc[green_mask].index
figs = []
for pat,fn in zip(['*_probe_dist_lo_*.pkl','*_probe_dist_hi_*.pkl','*_probe_dist_nofilt_*.pkl'],
                  ['probe_dist_lo','probe_dist_hi','probe_dist_nofilt']):
    figs.append(ave_and_plot_histograms(dfcs=dfcs,pat=pat,targets=targets,title='green_wt_' + fn))
figs[2]

# Figure 1G median

In [ ]:
def plot_lo_hi_medians_with_targets(df, targets, title=None):
    """
    df: pandas DataFrame with columns 'lo_state_median_position', 'hi_state_median_position'
    targets: dict like your example with 'lo' and 'hi' keys, each has keys:
             'pyasXPosition', 'pyasWidth', 'pyasState'
    """
    n = len(df)
    y_positions = np.arange(n)  # y-axis positions for each row

    fig = Figure(figsize=(6,6), dpi=300)
    canvas = FigureCanvas(fig)
    ax = fig.add_subplot(111)

    # Add target patches spanning full y range
    ymin, ymax = -0.5, n - 0.5  # so patches cover all rows vertically
    # for key in targets:
    #     targ = targets[key]
    #     rect = patches.Rectangle(
    #         (targ['pyasXPosition'], ymin),
    #         targ['pyasWidth'],
    #         ymax - ymin,
    #         facecolor='green' if targ['pyasState']=='lo' else 'red',
    #         alpha=0.3,
    #         label=f"Target {key}"
    #     )
    #     ax.add_patch(rect)


    for key in targets.keys():
        targ_dict = targets[key]
        tgt_clr = _force_clrs[0 if targ_dict['pyasState']=='lo' else 1]

        rect = patches.Rectangle(
            (targ_dict['pyasXPosition'], ymin),
            targ_dict['pyasWidth'],
            ymax - ymin,
            edgecolor=tgt_clr,
            facecolor=tgt_clr,
            label=f"Target {key}"
        )
        ax.add_patch(rect)        

    # Plot lines and dots for each row
    for i, (idx, row) in enumerate(df.iterrows()):
        x_lo = row['lo_state_median_position']
        x_hi = row['hi_state_median_position']
        if '210302_F1_C2' in row['parquet']:
            x_lo = x_lo-20
            x_hi = x_hi-20

        y = y_positions[i]

        # Plot dots
        ax.plot(x_lo, y, 'o', color='blue')
        ax.plot(x_hi, y, 'o', color='orange')

        # Connect dots with line
        ax.plot([x_lo, x_hi], [y, y], '-', color='gray')

        # Label with index to the right of hi dot
        # ax.text(x_hi + (max(df.max()) - min(df.min()))*0.01, y, str(idx),
        #         verticalalignment='center', fontsize=8)

    ax.set_ylim(ymin, ymax)
    ax.set_yticks(y_positions)
    ax.set_yticklabels(df.index)
    ax.set_xlabel('Median Position')
    ax.set_title('Lo and Hi State Median Positions per Row')
    ax.grid(axis='x', linestyle='--', alpha=0.5)
    ax.invert_yaxis()

    # Avoid duplicate labels in legend
    handles, labels = ax.get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    ax.legend(by_label.values(), by_label.keys())

    save_path = os.path.join('./Figure1', title + '.svg')
    fig.savefig(save_path,format = 'svg')
    print(f"Saved plot to {save_path}")

    return fig



In [ ]:
T = sinq.restore_table(dayflycell='250304_F3_C1')
targets = T.targets
# red_flies = sinq.df.loc[red_mask]
# blue_flies = sinq.df.loc[blue_mask]
# green_flies = sinq.df.loc[green_mask]
# fig = plot_lo_hi_medians_with_targets(pd.concat([red_flies,blue_flies,green_flies]), targets,title='lo_hi_median_for_all_flies')

fig = plot_lo_hi_medians_with_targets(sinq.df, targets,title='lo_hi_median_for_all_flies')

fig

## Figure 1H - fraction on vs. fraction on in off state

In [ ]:
import glob
import numpy as np
import pickle

# Define the path to the folder containing your pickle files

folder_path = "./Figure1/exports"

def calculate_fraction_from_histogram(file_path, target):

    print(target)
    with open(file_path, 'rb') as f:
        data = pickle.load(f)
        bins = data['probe_bins']
        counts = data['total_counts']
        
        # Handle the specific data correction if necessary
        if '210302_F1_C2' in file_path:
            counts = np.roll(counts, -10, axis=0)

    total_counts = counts.sum()
    if total_counts == 0:
        return 0.0

    binwidth = np.diff(bins[2:4])
    tbin_max = target['pyasXPosition']
    tbin_min = target['pyasXPosition'] + target['pyasWidth']

    target_bins = (bins[:-1] <= tbin_min+2*binwidth) & (bins[:-1] >= tbin_max-2*binwidth)

    ontarget = np.sum(counts[target_bins]) 

    return ontarget / total_counts

In [ ]:
T = sinq.df.loc['210917_F2_C1','Table'] # last one
targets = T.targets
# red_mask = (sinq.df['most_common_fiberLED']=='625_red')
# dfcs = sinq.df.loc[red_mask].index
dfcs = sinq.df.index
results_list = []

for target,pat in zip([targets['hi'],targets['lo']],['*_probe_dist_lo_*.pkl', '*_probe_dist_hi_*.pkl']):
    for dfc in dfcs:
        pattern = f'{dfc}'+pat
        file_path = glob.glob(os.path.join(folder_path, pattern))

        if file_path:
            file_path = file_path[0]
            fraction = calculate_fraction_from_histogram(file_path, target)
            
            # Append a dictionary with the results to the list
            results_list.append({
                'dfc': dfc,
                'pattern': pat,
                'target': target,
                'on_target_fraction': fraction
            })
        else:
            print(f"Warning: No file found for pattern {pattern}")

# Convert the list of dictionaries into a DataFrame
df_results = pd.DataFrame(results_list)

# Reshape the DataFrame to have patterns as columns
df_pivoted = df_results.pivot(index='dfc', columns='pattern', values='on_target_fraction')

print(df_pivoted)

In [ ]:
sinq.df.loc[df_pivoted.index,'lo_target_off_state'] = df_pivoted.loc[df_pivoted.index,'*_probe_dist_hi_*.pkl']
sinq.df.loc[df_pivoted.index,'hi_target_off_state'] = df_pivoted.loc[df_pivoted.index,'*_probe_dist_lo_*.pkl']

In [ ]:
df =  sinq.df.loc[:,['hi_state_on_target','lo_state_on_target','lo_target_off_state','hi_target_off_state']]

_force_clrs = [
    (np.float64(0.95447591), np.float64(0.47082238), np.float64(0.32310953)),
    (np.float64(0.7965014), np.float64(0.10506637), np.float64(0.31063031)),
    ]


fig = Figure(figsize=(4,12), dpi=300)
canvas = FigureCanvas(fig)
ax1 = fig.add_subplot(2,1,1)
ax2 = fig.add_subplot(2,1,2)

# Define the common x-axis positions and labels
x_positions = [0, 1]
x_labels = ['lo state', 'hi state']
x_labels_hi = ['hi state', 'lo state']

for idx, row in df.iterrows():
    random_color = np.random.rand(3)
    ax1.plot(x_positions, [row['lo_state_on_target'], row['lo_target_off_state']], color=random_color, alpha=1,label=row.name)
    ax1.scatter(x_positions, [row['lo_state_on_target'], row['lo_target_off_state']], c=['grey', 'grey'], s=50)

    ax2.plot(x_positions, [row['hi_state_on_target'], row['hi_target_off_state']], color=random_color, alpha=1,label=row.name)
    ax2.scatter(x_positions, [row['hi_state_on_target'], row['hi_target_off_state']], c=['grey', 'grey'], s=50)

ax1.set_title('Low Target')
ax1.set_ylabel('Fraction in Lo Target')
ax1.set_xticks(x_positions)
ax1.set_xticklabels(x_labels)
ax1.legend(loc='best')
ax1.set_ylim([0,1])

ax2.set_title('High Target')
ax2.set_ylabel('Fraction in Hi Target')
ax2.set_xticks(x_positions)
ax2.set_xticklabels(x_labels_hi)
ax2.set_ylim([0,1])

fig.savefig('./Figure1/fraction_in_target_on_vs_off_states.svg',format='svg')
fig